In [ ]:
# ══════════════════════════════════════════════════════════════
# PYSPARK — MapReduce clásico con conteo de palabras
# El "Hola Mundo" del Big Data
# ══════════════════════════════════════════════════════════════

!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# Arrancamos Spark
spark = SparkSession.builder \
    .appName("MapReduce — Conteo de Palabras") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("✅ Spark arrancado")

# ──────────────────────────────────────────────
# 1. LOS DATOS — Texto de ejemplo
# ──────────────────────────────────────────────
print("\n" + "="*60)
print("1. DATOS DE ENTRADA")
print("="*60)

# Simulamos un corpus de texto — en Big Data real serían
# millones de documentos leídos desde HDFS, S3, etc.
# Cada elemento de la lista es una línea de texto (como un fichero)
textos = [
    "el big data es una tecnología muy importante en la actualidad",
    "spark es un framework de big data muy potente y rápido",
    "el procesamiento distribuido permite analizar grandes volúmenes de datos",
    "map reduce es el patrón fundamental del procesamiento en big data",
    "spark mejora el rendimiento de map reduce usando memoria ram",
    "los datos son el nuevo petróleo de la economía digital",
    "el análisis de datos permite tomar mejores decisiones de negocio",
    "big data incluye tecnologías como spark hadoop kafka y mongodb",
    "el procesamiento paralelo distribuye el trabajo entre múltiples nodos",
    "spark permite procesar datos en tiempo real con spark streaming",
    "hadoop fue el primer framework popular de big data distribuido",
    "los datos estructurados y no estructurados se procesan con big data",
    "la inteligencia artificial necesita grandes volúmenes de datos para aprender",
    "spark sql permite usar sql para consultar grandes conjuntos de datos",
    "el almacenamiento distribuido es clave en arquitecturas de big data",
]

# Palabras que queremos ignorar por no aportar significado
# En procesamiento de lenguaje natural se llaman STOPWORDS
stopwords = {
    "el", "la", "los", "las", "de", "del", "en", "es", "y", "a",
    "un", "una", "que", "con", "por", "se", "son", "como", "muy",
    "para", "entre", "le", "lo", "al", "su", "sus", "o", "e"
}

print(f"Número de líneas de texto: {len(textos)}")
print(f"Número de stopwords:       {len(stopwords)}")
print("\nEjemplo de línea:")
print(f"  '{textos[0]}'")


# ──────────────────────────────────────────────
# 2. EL PATRÓN MAPREDUCE EXPLICADO
# ──────────────────────────────────────────────
print("\n" + "="*60)
print("2. EL PATRÓN MAPREDUCE — Concepto")
print("="*60)

print("""
MapReduce tiene 3 fases principales:

  FASE 1 — MAP:
  Cada línea de texto se divide en palabras.
  Cada palabra se convierte en un par (palabra, 1).
  Esta fase se ejecuta en PARALELO en todos los nodos.

  Entrada:  "spark es potente"
  Salida:   [("spark",1), ("es",1), ("potente",1)]

  FASE 2 — SHUFFLE (automático en Spark):
  Spark agrupa todos los pares con la misma clave.
  Todas las ocurrencias de "spark" van al mismo nodo.

  Entrada:  [("spark",1), ("spark",1), ("spark",1)]
  Salida:   ("spark", [1,1,1])

  FASE 3 — REDUCE:
  Se suman todos los valores de cada grupo.
  Esta fase también se ejecuta en paralelo por clave.

  Entrada:  ("spark", [1,1,1])
  Salida:   ("spark", 3)
""")


# ──────────────────────────────────────────────
# 3. IMPLEMENTACIÓN CON RDD (forma clásica)
# ──────────────────────────────────────────────
print("\n" + "="*60)
print("3. IMPLEMENTACIÓN CON RDD — MapReduce clásico")
print("="*60)

# Creamos el RDD con las líneas de texto
# Cada elemento del RDD es una línea completa
# Spark lo distribuye en particiones entre los núcleos disponibles
rdd_lineas = spark.sparkContext.parallelize(textos)

print(f"RDD creado con {rdd_lineas.count()} líneas")
print(f"Distribuido en {rdd_lineas.getNumPartitions()} particiones (núcleos)")

# ── FASE MAP ──
# flatMap() aplica la función a cada línea y aplana el resultado
# Sin flatMap tendríamos una lista de listas: [["spark","es"], ["big","data"]]
# Con flatMap tenemos una sola lista:          ["spark","es","big","data"]
rdd_palabras = rdd_lineas.flatMap(
    lambda linea: linea.lower().split(" ")
    # lower()   → convertimos a minúsculas para que "Spark" == "spark"
    # split(" ") → dividimos la línea en palabras por los espacios
)

print(f"\nTras flatMap — palabras totales: {rdd_palabras.count()}")
print(f"Ejemplo primeras palabras: {rdd_palabras.take(10)}")

# Filtramos stopwords y palabras vacías
rdd_filtrado = rdd_palabras.filter(
    lambda palabra: palabra not in stopwords  # excluimos stopwords
                    and len(palabra) > 2      # excluimos palabras muy cortas
                    and palabra.isalpha()     # excluimos números y símbolos
)

print(f"\nTras filtrar stopwords — palabras: {rdd_filtrado.count()}")

# Convertimos cada palabra en un par (palabra, 1)
# Este es el núcleo del patrón MAP: emitir (clave, valor)
rdd_pares = rdd_filtrado.map(lambda palabra: (palabra, 1))
# resultado: [("spark",1), ("datos",1), ("spark",1), ("big",1), ...]

print(f"\nEjemplo de pares (clave, valor) tras MAP:")
for par in rdd_pares.take(8):
    print(f"  {par}")

# ── FASE REDUCE ──
# reduceByKey() agrupa por clave y aplica la función a los valores
# Spark hace el SHUFFLE automáticamente (agrupa "spark" con "spark")
# y luego aplica la suma en paralelo
rdd_conteo = rdd_pares.reduceByKey(lambda a, b: a + b)
# resultado: [("spark", 5), ("datos", 7), ("big", 6), ...]

# Ordenamos por frecuencia descendente
rdd_ordenado = rdd_conteo.sortBy(lambda par: par[1], ascending=False)

print("\nTop 15 palabras más frecuentes (resultado del REDUCE):")
top15 = rdd_ordenado.take(15)
for palabra, count in top15:
    barra = "█" * count
    print(f"  {palabra:20s} {barra} ({count})")


# ──────────────────────────────────────────────
# 4. VARIACIONES DEL MAPREDUCE
# ──────────────────────────────────────────────
print("\n" + "="*60)
print("4. VARIACIONES — MapReduce más avanzado")
print("="*60)

# ── Variación 1: Longitud media de palabras por inicial
print("Longitud media de palabras agrupadas por letra inicial:")
rdd_longitud = rdd_filtrado \
    .map(lambda p: (p[0], len(p))) \
    .groupByKey() \
    .mapValues(lambda longitudes: round(sum(longitudes) / len(longitudes), 2)) \
    .sortByKey()

for letra, media in rdd_longitud.collect():
    print(f"  '{letra}' → longitud media: {media} caracteres")

# ── Variación 2: Coocurrencia — palabras que aparecen juntas en la misma línea
print("\nPares de palabras que coocurren en la misma línea (coocurrencia):")
def pares_coocurrencia(linea):
    # Generamos todos los pares posibles de palabras en la misma línea
    palabras = [p for p in linea.lower().split()
                if p not in stopwords and len(p) > 2 and p.isalpha()]
    pares = []
    for i in range(len(palabras)):
        for j in range(i+1, len(palabras)):
            # Ordenamos el par alfabéticamente para que
            # ("spark","datos") == ("datos","spark")
            par = tuple(sorted([palabras[i], palabras[j]]))
            pares.append((par, 1))
    return pares

rdd_cooc = rdd_lineas \
    .flatMap(pares_coocurrencia) \
    .reduceByKey(lambda a, b: a + b) \
    .filter(lambda x: x[1] >= 2) \
    .sortBy(lambda x: x[1], ascending=False)

print("Pares que aparecen juntos 2 o más veces:")
for par, count in rdd_cooc.take(8):
    print(f"  {par[0]:15s} + {par[1]:15s} → {count} veces")

# ── Variación 3: TF (Term Frequency) — frecuencia relativa de cada palabra
print("\nTF (Term Frequency) — frecuencia relativa de cada término:")
total_palabras = rdd_filtrado.count()
rdd_tf = rdd_conteo \
    .map(lambda par: (par[0], round(par[1] / total_palabras, 4))) \
    .sortBy(lambda x: x[1], ascending=False)

print(f"Total palabras (sin stopwords): {total_palabras}")
print("Top 10 por TF:")
for palabra, tf in rdd_tf.take(10):
    print(f"  {palabra:20s} → TF = {tf}")


# ──────────────────────────────────────────────
# 5. IMPLEMENTACIÓN CON DATAFRAME (forma moderna)
# ──────────────────────────────────────────────
print("\n" + "="*60)
print("5. IMPLEMENTACIÓN CON DATAFRAME — Forma moderna")
print("="*60)

print("""
En Spark moderno se prefiere usar DataFrames en lugar de RDDs
porque el optimizador de Spark (Catalyst) los optimiza automáticamente.
El resultado es el mismo pero el código es más legible y eficiente.
""")

# Creamos un DataFrame con las líneas de texto
df_texto = spark.createDataFrame(
    [(i, linea) for i, linea in enumerate(textos)],
    ["id_linea", "texto"]
)

# split() divide el texto en palabras (array)
# explode() convierte el array en filas individuales (como flatMap)
# lower() convierte a minúsculas
df_palabras = df_texto \
    .withColumn("palabra", F.explode(F.split(F.lower(F.col("texto")), " "))) \
    .filter(~F.col("palabra").isin(list(stopwords))) \
    .filter(F.length("palabra") > 2) \
    .filter(F.col("palabra").rlike("^[a-záéíóúñ]+$"))

# groupBy + count() hace el MapReduce internamente
df_conteo = df_palabras \
    .groupBy("palabra") \
    .count() \
    .orderBy("count", ascending=False)

print("Top 15 palabras (con DataFrame):")
df_conteo.show(15)


# ──────────────────────────────────────────────
# 6. VISUALIZACIONES
# ──────────────────────────────────────────────
print("\n" + "="*60)
print("6. VISUALIZACIONES")
print("="*60)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F8F9FA',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('MapReduce — Análisis de Texto con PySpark',
             fontsize=15, fontweight='bold')

# ── Gráfico 1: Top palabras (barras horizontales)
top20 = rdd_ordenado.take(20)
palabras_top = [p[0] for p in top20]
conteos_top  = [p[1] for p in top20]

colores = plt.cm.Blues(np.linspace(0.4, 0.9, len(palabras_top)))[::-1]
bars = axes[0,0].barh(palabras_top[::-1], conteos_top[::-1],
                       color=colores, edgecolor='white', linewidth=1.2)
axes[0,0].set_title('Top 20 palabras más frecuentes', fontweight='bold')
axes[0,0].set_xlabel('Frecuencia')
for bar, val in zip(bars, conteos_top[::-1]):
    axes[0,0].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                   str(val), va='center', fontsize=9, fontweight='bold')

# ── Gráfico 2: Distribución de frecuencias
todos_conteos = [p[1] for p in rdd_ordenado.collect()]
axes[0,1].hist(todos_conteos, bins=8, color='#1B4F72', edgecolor='white',
               linewidth=1.5, alpha=0.85)
axes[0,1].set_title('Distribución de frecuencias\n(Ley de Zipf)', fontweight='bold')
axes[0,1].set_xlabel('Frecuencia de la palabra')
axes[0,1].set_ylabel('Número de palabras')
axes[0,1].axvline(x=np.mean(todos_conteos), color='#C0392B', linestyle='--',
                   linewidth=2, label=f'Media: {np.mean(todos_conteos):.1f}')
axes[0,1].legend()

# ── Gráfico 3: TF de las top 10 palabras
top10_tf = rdd_tf.take(10)
palabras_tf = [p[0] for p in top10_tf]
valores_tf  = [p[1] for p in top10_tf]

colores_tf = plt.cm.Greens(np.linspace(0.4, 0.9, len(palabras_tf)))[::-1]
bars3 = axes[1,0].bar(range(len(palabras_tf)), valores_tf,
                       color=colores_tf, edgecolor='white', linewidth=1.5)
axes[1,0].set_title('Term Frequency (TF) — Top 10', fontweight='bold')
axes[1,0].set_ylabel('TF (frecuencia relativa)')
axes[1,0].set_xticks(range(len(palabras_tf)))
axes[1,0].set_xticklabels(palabras_tf, rotation=30, ha='right')
for bar, val in zip(bars3, valores_tf):
    axes[1,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                   f'{val:.3f}', ha='center', fontsize=8, fontweight='bold')

# ── Gráfico 4: Diagrama del flujo MapReduce
axes[1,1].axis('off')
axes[1,1].set_title('Flujo MapReduce', fontweight='bold')

etapas = ['ENTRADA\n(líneas de texto)', 'MAP\n(palabra, 1)',
          'SHUFFLE\n(agrupación)', 'REDUCE\n(palabra, N)']
ejemplos = ['"spark es potente"', '("spark",1)\n("es",1)\n("potente",1)',
            '"spark" → [1,1,1,1]\n"datos" → [1,1,1]',
            '"spark" → 4\n"datos" → 3']
colores_etapas = ['#1B4F72','#17A589','#F39C12','#C0392B']

for i, (etapa, ejemplo, color) in enumerate(zip(etapas, ejemplos, colores_etapas)):
    x = 0.05 + i * 0.24
    rect = plt.Rectangle((x, 0.55), 0.2, 0.3, facecolor=color,
                          alpha=0.85, transform=axes[1,1].transAxes)
    axes[1,1].add_patch(rect)
    axes[1,1].text(x + 0.1, 0.72, etapa, transform=axes[1,1].transAxes,
                   ha='center', va='center', fontsize=8.5, color='white',
                   fontweight='bold')
    axes[1,1].text(x + 0.1, 0.35, ejemplo, transform=axes[1,1].transAxes,
                   ha='center', va='center', fontsize=7.5, color='#2C3E50',
                   style='italic',
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='#EBF5FB',
                             edgecolor=color, alpha=0.8))
    if i < 3:
        axes[1,1].annotate('', xy=(x + 0.24, 0.7), xytext=(x + 0.2, 0.7),
                            xycoords='axes fraction', textcoords='axes fraction',
                            arrowprops=dict(arrowstyle='->', color='#566573', lw=2))

plt.tight_layout()
plt.savefig('mapreduce_palabras.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Notebook completo")
print("\nResumen de conceptos vistos:")
print("  · flatMap()       → divide líneas en palabras (fase MAP)")
print("  · filter()        → elimina stopwords y palabras cortas")
print("  · map(x → (x,1)) → emite pares (clave, valor)")
print("  · reduceByKey()   → suma por clave (fase REDUCE)")
print("  · sortBy()        → ordena por frecuencia")
print("  · groupByKey()    → agrupa valores por clave")
print("  · coocurrencia    → palabras que aparecen juntas")
print("  · TF              → frecuencia relativa de cada término")
print("  · DataFrame       → forma moderna equivalente al RDD")

spark.stop()